In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Climate Data QC Pipeline - Hybrid Approach for Indonesian Rainfall
• Temperature: IQR x 3.0 (standard for near-normal distribution)
• Rainfall:    Hybrid (99.9th percentile + physical limit 600 mm/day)
               More relevant for skewed tropical rainfall distribution

Robust handling for stations with missing parameters:
  - Parameter dianggap "tidak tersedia" jika SEMUA nilai = NaN
  - QC hanya dijalankan untuk parameter yang tersedia (minimal 1 nilai valid)
  - Plot hanya di-generate untuk parameter yang tersedia
  - Metadata ketersediaan parameter disimpan di output QC
"""

import os
import sys
import glob
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

# ==============================================================================
# SETUP PROJECT STRUCTURE
# ==============================================================================

workDir = os.getcwd()
dataDir = os.path.join('..', '01.data')
outDir = os.path.join('..', '02.output')
srcDir = os.path.join('00.src')

obsDir = os.path.join(dataDir, "01.Obs_Homo")          # Data observasi mentah
qcDir = os.path.join(dataDir, "02.Obs_HybridIQR")       # Output hasil QC
reportDir = os.path.join(dataDir, "03.Report")    # Report visualisasi

# Create directory structure
for directory in [dataDir, outDir, srcDir, obsDir, qcDir, reportDir]:
    os.makedirs(directory, exist_ok=True)

# Setup logging
log_file = os.path.join(reportDir, 'qc_processing.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ==============================================================================
# QC BOUNDARY CALCULATION - HYBRID APPROACH FOR RAINFALL
# ==============================================================================

# Konstanta fisik untuk Indonesia
RAINFALL_PHYSICAL_MAX = 650.0  # mm/hari - rekor absolut Indonesia
RAINFALL_PHYSICAL_MIN = 0.0    # mm/hari - constraint fisik

def iqr_up(df, col, iqrx):
    """Hitung batas atas IQR"""
    valid_data = df[col].dropna()
    if len(valid_data) == 0:
        return np.nan
    Q1 = valid_data.quantile(0.25)
    Q3 = valid_data.quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + iqrx * IQR
    return upper

def iqr_low(df, col, iqrx):
    """Hitung batas bawah IQR (clamp ke 0 untuk curah hujan)"""
    valid_data = df[col].dropna()
    if len(valid_data) == 0:
        return np.nan
    Q1 = valid_data.quantile(0.25)
    Q3 = valid_data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - iqrx * IQR
    if col == 'RAINFALL_24H_MM':
        lower = max(RAINFALL_PHYSICAL_MIN, lower)
    return lower

def rainfall_upper_bound(df, col='RAINFALL_24H_MM', percentile=99.9):
    """
    Hitung batas atas curah hujan dengan pendekatan hybrid:
    1. Percentile tinggi dari data historis stasiun
    2. Dibatasi oleh batas fisik maksimum Indonesia (650 mm/hari)
    """
    valid_rain = df[df[col] > 0][col].dropna()
    if len(valid_rain) == 0:
        return RAINFALL_PHYSICAL_MAX
    
    percentile_bound = valid_rain.quantile(percentile / 100.0)
    upper_bound = min(percentile_bound, RAINFALL_PHYSICAL_MAX)
    upper_bound = max(upper_bound, 450.0)  # Minimal 450 mm untuk event ekstrem
    return upper_bound

def rangeClean(df, col, method='IQR', iqrx=3, sigma=4, rainfall_percentile=99.9):
    """
    Bersihkan data dengan metode spesifik per parameter.
    Returns: DataFrame dengan kolom QC flags dan nilai outlier di-mask menjadi NaN
    """
    data = df.copy()
    
    cols_to_process = [col] if isinstance(col, str) else col
    
    for colName in cols_to_process:
        if colName not in data.columns:
            logger.warning(f"  ⚠ Kolom '{colName}' tidak ditemukan di data - dilewati")
            continue
        
        # Cek apakah parameter tersedia (minimal 1 nilai non-NaN)
        valid_count = data[colName].notna().sum()
        if valid_count == 0:
            logger.warning(f"  ⚠ Parameter '{colName}' tidak tersedia (semua nilai NaN) - dilewati")
            # Tandai sebagai tidak tersedia di output
            data[f'{colName}_AVAILABLE'] = False
            data[f'{colName}_UP'] = np.nan
            data[f'{colName}_LOW'] = np.nan
            data[f'{colName}_QC_METHOD'] = 'NOT_AVAILABLE'
            continue
        
        data[f'{colName}_AVAILABLE'] = True
        
        # === QC UNTUK SUHU (IQR) ===
        if colName in ['TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C']:
            upper = iqr_up(data, colName, iqrx)
            lower = iqr_low(data, colName, iqrx)
            qc_method = f'IQR x {iqrx}'
        
        # === QC UNTUK CURAH HUJAN (HYBRID) ===
        elif colName == 'RAINFALL_24H_MM':
            lower = RAINFALL_PHYSICAL_MIN
            upper = rainfall_upper_bound(data, col=colName, percentile=rainfall_percentile)
            qc_method = f'Hybrid (P{rainfall_percentile} + {RAINFALL_PHYSICAL_MAX}mm)'
        
        else:
            # Fallback untuk parameter lain
            upper = iqr_up(data, colName, iqrx)
            lower = iqr_low(data, colName, iqrx)
            qc_method = f'IQR x {iqrx}'
        
        # Simpan batas QC dan metode
        data[f'{colName}_UP']        = upper
        data[f'{colName}_LOW']       = lower
        data[f'{colName}_QC_METHOD'] = qc_method
        
        # Mask outlier menjadi NaN (hanya jika batas valid)
        if not (pd.isna(upper) or pd.isna(lower)):
            mask = (data[colName] < lower) | (data[colName] > upper)
            outlier_count = mask.sum()
            if outlier_count > 0:
                data.loc[mask, colName] = np.nan
                logger.debug(f"    → {outlier_count} outlier di-mask untuk {colName}")
    
    return data

# ==============================================================================
# VISUALISASI QC
# ==============================================================================

def plot1D(df, x, y, tittle, x_tittle, y_tittle, ax, 
           bottom=None, top=None, tick_interval=None, 
           upper=None, lower=None, show_bounds=True, 
           qc_method='IQR', iqrx=3, percentile=99.9,
           show_tittle=True, show_xtittle=True, show_ytittle=True, color=None):
    """Plot time series dengan visualisasi QC boundaries"""
    # Skema warna konsisten
    if color is None:
        if y == "TEMPERATURE_AVG_C":
            color = "#ffc107"
        elif y == "TEMP_24H_TN_C":
            color = "#007bff"
        elif y == "TEMP_24H_TX_C":
            color = "#dc3545"
        else:
            color = "#28a745"
    # Validasi data kosong
    if df.empty or df[y].isna().all():
        ax.text(0.5, 0.5, 'No data available', 
                transform=ax.transAxes, ha='center', va='center', 
                fontsize=8, color='gray', fontname='Noto Sans')
        if show_tittle:
            ax.set_title(tittle, fontsize=10, fontweight='bold', fontname='Noto Sans')
        ax.set_axis_off()
        return
    
    # Hitung batas QC jika belum diberikan
    if upper is None or lower is None:
        if y == 'RAINFALL_24H_MM':
            lower = RAINFALL_PHYSICAL_MIN
            upper = rainfall_upper_bound(df, col=y, percentile=percentile)
            bound_label = f'P{percentile} + {RAINFALL_PHYSICAL_MAX}mm'
        else:
            upper = iqr_up(df, col=y, iqrx=iqrx)
            lower = iqr_low(df, col=y, iqrx=iqrx)
            bound_label = f'{iqrx}xIQR'
    else:
        bound_label = qc_method
    
    # Tentukan batas sumbu Y
    data_min = df[y].min()
    data_max = df[y].max()
    
    if bottom is None:
        bottom = min(data_min, lower) - 0.05 * (data_max - data_min) if not pd.isna(lower) else data_min - 0.05 * (data_max - data_min)
    if top is None:
        top = max(data_max, upper) + 0.05 * (data_max - data_min) if not pd.isna(upper) else data_max + 0.05 * (data_max - data_min)
    
    # Plot data utama
    sns.lineplot(data=df, x=x, y=y, errorbar=None, linewidth=0.65, 
                 ax=ax, color=color, alpha=0.85, zorder=2)
    
    # Tandai outlier
    if upper is not None and lower is not None and not (pd.isna(upper) or pd.isna(lower)):
        outliers = df[(df[y] < lower) | (df[y] > upper)]
        if not outliers.empty:
            ax.scatter(outliers[x], outliers[y], 
                      color='red', s=15, marker='x', alpha=0.8, 
                      label='Outliers', zorder=3, linewidth=0.8)
    
    # Set judul dan label
    if show_tittle:
        ax.set_title(tittle, fontsize=10, fontweight='bold', fontname='Noto Sans')
    if show_xtittle:
        ax.set_xlabel(x_tittle, fontsize=6, fontweight='semibold', fontname='Noto Sans')
    else:
        ax.set_xlabel('')
    if show_ytittle:
        ax.set_ylabel(y_tittle, fontsize=6, fontweight='semibold', fontname='Noto Sans')
    else:
        ax.set_ylabel('')
    
    # Format ticks
    ax.tick_params(axis='x', labelsize=5, labelrotation=0, 
                   labelcolor='black', width=0.8, length=3)
    ax.tick_params(axis='y', labelsize=5, 
                   labelcolor='black', width=0.8, length=3)
    
    # Garis batas QC
    if show_bounds and upper is not None and lower is not None and not (pd.isna(upper) or pd.isna(lower)):
        ax.axhline(y=upper, color='#1e3a8a', linestyle='--', 
                  label=f'Upper: {upper:.1f} ({bound_label})', linewidth=0.9, alpha=0.85, zorder=1)
        if lower > 0 or y != 'RAINFALL_24H_MM':
            ax.axhline(y=lower, color='#b91c1c', linestyle='--', 
                      label=f'Lower: {lower:.1f}', linewidth=0.9, alpha=0.85, zorder=1)
        ax.legend(fontsize=5.5, loc='upper right', ncol=1, 
                 frameon=True, framealpha=0.9, edgecolor='gray', 
                 handletextpad=0.3, columnspacing=0.5)
    
    # Set batas sumbu Y
    ax.set_ylim(bottom=bottom, top=top)
    
    # Interval grid adaptif
    if tick_interval is None:
        y_range = top - bottom
        if y_range > 100:
            tick_interval = 20
        elif y_range > 50:
            tick_interval = 10
        elif y_range > 20:
            tick_interval = 5
        elif y_range > 10:
            tick_interval = 2
        else:
            tick_interval = 1
    
    ax.yaxis.set_major_locator(MultipleLocator(tick_interval))
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    
    # Grid
    ax.margins(x=0.01, y=0.05)
    ax.grid(True, which='major', linewidth=0.5, alpha=0.3, color='gray')
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.15, linestyle=':')

def TimeSeriesPlot(df, available_params, savefig=True, outPath=None):
    """
    Plot time series hanya untuk parameter yang tersedia
    available_params: dict dengan kunci 'temperature' (list) dan 'rainfall' (bool)
    """
    # Tentukan jumlah subplot berdasarkan parameter yang tersedia
    n_plots = len(available_params['temperature']) + (1 if available_params['rainfall'] else 0)
    if n_plots == 0:
        logger.warning("Tidak ada parameter tersedia untuk plotting")
        return None
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(8, 2 * n_plots), dpi=300, sharex=True)
    if n_plots == 1:
        axes = [axes]
    
    fig.suptitle("TIME SERIES WITH PARAMETER-SPECIFIC QC METHODS", 
                fontsize=11, fontweight='bold', fontname='Noto Sans', y=0.995)
    
    plot_idx = 0
    
    # Plot suhu yang tersedia
    temp_order = ['TEMPERATURE_AVG_C', 'TEMP_24H_TX_C', 'TEMP_24H_TN_C']
    for temp_col in temp_order:
        if temp_col in available_params['temperature']:
            if temp_col == 'TEMPERATURE_AVG_C':
                title = 'Temperature Average (IQR x 3.0)'
                y_label = 'T Ave (°C)'
                iqrx_val = 3.0
            elif temp_col == 'TEMP_24H_TX_C':
                title = 'Temperature Maximum (IQR x 3.0)'
                y_label = 'T Max (°C)'
                iqrx_val = 3.0
            else:  # TEMP_24H_TN_C
                title = 'Temperature Minimum (IQR x 3.0)'
                y_label = 'T Min (°C)'
                iqrx_val = 3.0
            
            plot1D(df, x='time', y=temp_col, 
                   tittle=title, x_tittle='', y_tittle=y_label, ax=axes[plot_idx],
                   show_tittle=False, show_xtittle=(plot_idx == n_plots - 1),
                   show_bounds=True, qc_method='IQR', iqrx=iqrx_val)
            plot_idx += 1
    
    # Plot curah hujan jika tersedia
    if available_params['rainfall']:
        plot1D(df, x='time', y='RAINFALL_24H_MM', 
               tittle='Precipitation (Hybrid: P99.9 + 600mm)', 
               x_tittle='Year', y_tittle='CH (mm)', ax=axes[plot_idx],
               show_tittle=False, show_xtittle=True,
               show_bounds=True, qc_method='Hybrid', percentile=99.9)
    
    for ax in axes:
        ax.xaxis.set_major_locator(plt.MaxNLocator(10))
    
    plt.tight_layout(rect=[0, 0.01, 1, 0.98])
    
    if savefig:
        if outPath is None:
            outPath = 'time_series_plot.png'
        Path(outPath).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(outPath, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return outPath
    else:
        plt.show()
        return None

def ComparisonPlot(df, df_qc, x, y, suptitle='RAW vs QC COMPARISON', 
                   tittle1='Raw Data', tittle2='QC Data',
                   x_tittle='Year', y_tittle1='Raw Value', y_tittle2='QC Value',
                   savefig=True, outPath=None):
    """Bandingkan data mentah vs QC"""
    if f'{y}_UP' in df_qc.columns and f'{y}_LOW' in df_qc.columns:
        upper = df_qc[f'{y}_UP'].iloc[0]
        lower = df_qc[f'{y}_LOW'].iloc[0]
    else:
        if y == 'RAINFALL_24H_MM':
            lower = RAINFALL_PHYSICAL_MIN
            upper = rainfall_upper_bound(df, col=y, percentile=99)
        else:
            upper = iqr_up(df, col=y, iqrx=3.0)
            lower = iqr_low(df, col=y, iqrx=3.0)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), dpi=300, sharex=True)
    fig.suptitle(suptitle, fontsize=11, fontweight='bold', fontname='Noto Sans', y=0.99)
    
    # Plot raw data
    plot1D(df, x=x, y=y, tittle=tittle1, x_tittle='', y_tittle=y_tittle1, ax=ax1,
           show_tittle=False, show_xtittle=False,
           upper=upper, lower=lower,
           show_bounds=True, 
           qc_method='Hybrid' if y == 'RAINFALL_24H_MM' else 'IQR',
           iqrx=3.0, percentile=99)
    
    # Plot QC data
    plot1D(df_qc, x=x, y=y, tittle=tittle2, x_tittle=x_tittle, y_tittle=y_tittle2, ax=ax2,
           show_tittle=False, show_xtittle=True,
           upper=upper, lower=lower,
           show_bounds=True,
           qc_method='Hybrid' if y == 'RAINFALL_24H_MM' else 'IQR',
           iqrx=3.0, percentile=99)
    
    plt.tight_layout(rect=[0, 0.01, 1, 0.97])
    
    if savefig:
        if outPath is None:
            outPath = 'comparison_plot.png'
        Path(outPath).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(outPath, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        return outPath
    else:
        plt.show()
        return None

# ==============================================================================
# QC PROCESSING PIPELINE DENGAN PENANGANAN PARAMETER TIDAK TERSEDIA
# ==============================================================================

def process_station_file(filepath: str, qc_dir: str, report_dir: str) -> bool:
    """
    Proses QC dengan penanganan robust untuk parameter tidak tersedia:
      - Parameter dianggap "tidak tersedia" jika SEMUA nilai = NaN
      - QC hanya dijalankan untuk parameter dengan minimal 1 nilai valid
    """
    try:
        filename = os.path.basename(filepath)
        parts = filename.replace('.csv', '').split('_')
        
        if len(parts) < 7:
            logger.error(f"Format filename tidak valid: {filename}")
            return False
            
        # Format filename: FKLIM_DAILY_{wmo_id}_{start}_{end}.csv
        wmo_id = parts[7]
        start_date_raw = parts[5]
        end_date_raw = parts[6]
        start_year = start_date_raw.split('-')[0]
        end_year = end_date_raw.split('-')[0]
        
        logger.info(f"Memproses stasiun {wmo_id} | Periode: {start_date_raw} - {end_date_raw}")
        df = pd.read_csv(filepath, parse_dates=['time'])
        
        if df.empty:
            logger.warning(f"File kosong: {filename}")
            return False
        
        # Validasi kolom waktu minimal
        if 'time' not in df.columns:
            logger.error(f"Kolom 'time' tidak ditemukan di {filename}")
            return False
        
        # Isi forward metadata stasiun (jika ada)
        metadata_cols = ['wmo_id', 'name', 'latitude', 'longitude']
        for col in metadata_cols:
            if col in df.columns:
                df[col] = df[col].ffill()
        
        station_name = df['name'].iloc[0] if 'name' in df.columns else f"Station_{wmo_id}"
        logger.info(f"  Stasiun: {station_name}")
        # Setup direktori output
        safe_name   = str(station_name).replace(' ', '_').replace('/', '_').replace('\\', '_')
        fig_out_dir = Path(report_dir) / f'{str(wmo_id)}_{safe_name}'
        fig_out_dir.mkdir(parents=True, exist_ok=True)
        
        # ======================================================================
        # PENENTUAN KETERSEDIAAN PARAMETER (KRUSIAL!)
        # Parameter dianggap "tidak tersedia" jika SEMUA nilai = NaN
        # ======================================================================
        available_params = {
            'temperature': [],
            'rainfall': False,
            'all_available': False
        }
        
        # Cek parameter suhu
        temp_cols = ['TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C']
        for col in temp_cols:
            if col in df.columns and df[col].notna().sum() > 0:
                available_params['temperature'].append(col)
                logger.debug(f"  ✓ Parameter {col} tersedia ({df[col].notna().sum()} nilai valid)")
            elif col in df.columns:
                logger.warning(f"  ⚠ Parameter {col} tidak tersedia (semua nilai NaN)")
            else:
                logger.warning(f"  ⚠ Kolom {col} tidak ditemukan di data")
        
        # Cek curah hujan
        if 'RAINFALL_24H_MM' in df.columns and df['RAINFALL_24H_MM'].notna().sum() > 0:
            available_params['rainfall'] = True
            logger.debug(f"  ✓ Parameter RAINFALL_24H_MM tersedia ({df['RAINFALL_24H_MM'].notna().sum()} nilai valid)")
        elif 'RAINFALL_24H_MM' in df.columns:
            logger.warning(f"  ⚠ Parameter RAINFALL_24H_MM tidak tersedia (semua nilai NaN)")
        else:
            logger.warning(f"  ⚠ Kolom RAINFALL_24H_MM tidak ditemukan di data")
        
        available_params['all_available'] = (len(available_params['temperature']) > 0 or available_params['rainfall'])
        
        if not available_params['all_available']:
            logger.error(f"✗ Stasiun {wmo_id} tidak memiliki parameter yang dapat diproses (semua parameter NaN atau tidak ada)")
            return False
        
        logger.info(f"  Parameter tersedia: {len(available_params['temperature'])} suhu, {'ada' if available_params['rainfall'] else 'tidak ada'} curah hujan")
        
        # ======================================================================
        # PROSES QC HANYA UNTUK PARAMETER YANG TERSEDIA
        # ======================================================================
        data_qc = df.copy()
        
        # QC untuk parameter suhu yang tersedia
        if available_params['temperature']:
            try:
                temp_qc = rangeClean(
                    df,
                    col=available_params['temperature'],
                    method='IQR',
                    iqrx=3.0
                )
                # Salin hasil QC ke data utama
                for col in available_params['temperature']:
                    if col in temp_qc.columns:
                        data_qc[col] = temp_qc[col]
                        for qc_col in [f'{col}_UP', f'{col}_LOW', f'{col}_QC_METHOD', f'{col}_AVAILABLE']:
                            if qc_col in temp_qc.columns:
                                data_qc[qc_col] = temp_qc[qc_col]
                logger.info(f"✓ QC temperature selesai untuk {len(available_params['temperature'])} parameter")
            except Exception as e:
                logger.error(f"✗ Gagal QC temperature: {str(e)}")
                return False
        
        # QC untuk curah hujan jika tersedia
        if available_params['rainfall']:
            try:
                rain_qc = rangeClean(
                    df,
                    col='RAINFALL_24H_MM',
                    method='HYBRID',
                    rainfall_percentile=99.9
                )
                col = 'RAINFALL_24H_MM'
                if col in rain_qc.columns:
                    data_qc[col] = rain_qc[col]
                    for qc_col in [f'{col}_UP', f'{col}_LOW', f'{col}_QC_METHOD', f'{col}_AVAILABLE']:
                        if qc_col in rain_qc.columns:
                            data_qc[qc_col] = rain_qc[qc_col]
                logger.info(f"✓ QC rainfall selesai")
            except Exception as e:
                logger.error(f"✗ Gagal QC rainfall: {str(e)}")
                return False
        
        # ======================================================================
        # SIMPAN METADATA KETERSEDIAAN KE OUTPUT
        # ======================================================================
        data_qc['PARAM_TEMPERATURE_AVAILABLE'] = len(available_params['temperature']) > 0
        data_qc['PARAM_RAINFALL_AVAILABLE']    = available_params['rainfall']
        data_qc['PARAM_COUNT_AVAILABLE']       = len(available_params['temperature']) + (1 if available_params['rainfall'] else 0)
        
        # ======================================================================
        # GENERATE VISUALISASI
        # ======================================================================
        qc_filename = f'FKLIM_QC_DAILY_HYBRID_IQR_{start_date_raw}_{end_date_raw}_{wmo_id}.csv'
        qc_filepath = Path(qc_dir) / qc_filename
        
        # Time Series Plot (hanya parameter tersedia)
        ts_figname = f'01_PLOT_TIME_SERIES_{safe_name.upper()}_{start_year}-{end_year}.png'
        try:
            TimeSeriesPlot(df=data_qc, available_params=available_params, 
                          savefig=True, outPath=str(fig_out_dir / ts_figname))
            logger.info(f"✓ Time series plot tersimpan: {ts_figname}")
        except Exception as e:
            logger.warning(f"⚠ Gagal generate time series plot: {str(e)}")
        
        # Comparison Plots (hanya untuk parameter yang diproses QC)
        plot_configs = []
        if 'TEMPERATURE_AVG_C' in available_params['temperature']:
            plot_configs.append(('TEMPERATURE_AVG_C', 'T Ave (°C)', 'TEMPERATURE AVERAGE', 
                               f'02_PLOT_COMPARISON_TEMPERATURE_AVERAGE_{safe_name.upper()}_{start_year}-{end_year}.png', '°C'))
        if 'TEMP_24H_TN_C' in available_params['temperature']:
            plot_configs.append(('TEMP_24H_TN_C', 'T Min (°C)', 'TEMPERATURE MINIMUM',
                               f'03_PLOT_COMPARISON_TEMPERATURE_MINIMUM_{safe_name.upper()}_{start_year}-{end_year}.png', '°C'))
        if 'TEMP_24H_TX_C' in available_params['temperature']:
            plot_configs.append(('TEMP_24H_TX_C', 'T Max (°C)', 'TEMPERATURE MAXIMUM',
                               f'04_PLOT_COMPARISON_TEMPERATURE_MAXIMUM_{safe_name.upper()}_{start_year}-{end_year}.png', '°C'))
        if available_params['rainfall']:
            plot_configs.append(('RAINFALL_24H_MM', 'CH (mm)', 'PRECIPITATION (HYBRID)',
                               f'05_PLOT_COMPARISON_CURAH_HUJAN_{safe_name.upper()}_{start_year}-{end_year}.png', 'mm'))
        
        for col, y_title, suptitle, fig_name, unit in plot_configs:
            try:
                ComparisonPlot(
                    df=df, 
                    df_qc=data_qc, 
                    x='time', 
                    y=col,
                    suptitle=f'RAW vs CLEAN {suptitle}',
                    tittle1='Raw Data',
                    tittle2='QC Data',
                    x_tittle='Tahun',
                    y_tittle1=f'Raw {y_title}',
                    y_tittle2=f'QC {y_title}',
                    savefig=True,
                    outPath=str(fig_out_dir / fig_name)
                )
                logger.info(f"✓ Comparison plot tersimpan: {fig_name}")
            except Exception as e:
                logger.warning(f"⚠ Gagal generate plot {col}: {str(e)}")
        
        # ======================================================================
        # LOGGING STATISTIK OUTLIER
        # ======================================================================
        logger.info("  Statistik QC:")
        all_params = available_params['temperature'] + (['RAINFALL_24H_MM'] if available_params['rainfall'] else [])
        for col in all_params:
            if col in df.columns and col in data_qc.columns:
                raw_valid = df[col].notna().sum()
                qc_valid = data_qc[col].notna().sum()
                outliers = raw_valid - qc_valid
                if raw_valid > 0:
                    pct = (outliers / raw_valid) * 100
                    method = data_qc[f'{col}_QC_METHOD'].iloc[0] if f'{col}_QC_METHOD' in data_qc.columns else 'N/A'
                    logger.info(f"    - {col} ({method}): {outliers} outlier ({pct:.2f}%) dari {raw_valid} nilai valid")
                else:
                    logger.info(f"    - {col}: Tidak ada nilai valid untuk diproses")
        
        # ======================================================================
        # SIMPAN HASIL QC
        # ======================================================================
        try:
            data_qc.to_csv(qc_filepath, index=False, encoding='utf-8')
            logger.info(f"✓ Hasil QC tersimpan: {qc_filename}")
            logger.info(f"  - Parameter diproses: {len(available_params['temperature'])} suhu, {1 if available_params['rainfall'] else 0} curah hujan")
            logger.info(f"  - Total observasi: {len(data_qc)}")
            
            return True
        except Exception as e:
            logger.error(f"✗ Gagal simpan hasil QC: {str(e)}")
            return False
            
    except Exception as e:
        logger.exception(f"✗ Error kritis memproses {filepath}: {str(e)}")
        return False
    finally:
        plt.close('all')

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

if __name__ == "__main__":
    logger.info("="*70)
    logger.info("MEMULAI PROSES QC HARIAN - PENANGANAN PARAMETER TIDAK TERSEDIA")
    logger.info("  • Parameter dianggap 'tidak tersedia' jika SEMUA nilai = NaN")
    logger.info("  • QC hanya dijalankan untuk parameter dengan minimal 1 nilai valid")
    logger.info("  • Suhu: IQR x 3.0 | Curah Hujan: Hybrid (P99.9 + 600mm)")
    logger.info(f"Input directory : {obsDir}")
    logger.info(f"QC output       : {qcDir}")
    logger.info(f"Report directory: {reportDir}")
    logger.info(f"Log file        : {log_file}")
    logger.info("="*70)
    
    clean_paths = sorted(glob.glob(os.path.join(obsDir, 'FKLIM_QC_DAILY_*.csv')))
    
    if not clean_paths:
        logger.error("Tidak ditemukan file CSV di direktori input!")
        logger.error(f"Cari di: {obsDir}")
        logger.error("Format yang dicari: FKLIM_QC_DAILY_*.csv")
        sys.exit(1)
    
    logger.info(f"Ditemukan {len(clean_paths)} file untuk diproses")
    
    success_count = 0
    for i, filepath in enumerate(clean_paths, 1):
        logger.info(f"\n[{i}/{len(clean_paths)}] {'='*60}")
        if process_station_file(filepath, qcDir, reportDir):
            success_count += 1
    
    logger.info("\n" + "="*70)
    logger.info("RINGKASAN EKSEKUSI QC")
    logger.info("="*70)
    logger.info(f"Total file diproses : {len(clean_paths)}")
    logger.info(f"Sukses              : {success_count}")
    logger.info(f"Gagal               : {len(clean_paths) - success_count}")
    logger.info("Kebijakan penanganan parameter:")
    logger.info("  • Parameter dengan SEMUA nilai NaN → dianggap 'tidak tersedia'")
    logger.info("  • QC hanya dijalankan untuk parameter dengan ≥1 nilai valid")
    logger.info("  • Plot hanya di-generate untuk parameter yang tersedia")
    logger.info("  • Metadata ketersediaan disimpan di kolom output:")
    logger.info("      - PARAM_TEMPERATURE_AVAILABLE")
    logger.info("      - PARAM_RAINFALL_AVAILABLE")
    logger.info("      - PARAM_COUNT_AVAILABLE")
    logger.info(f"Log detail          : {log_file}")
    logger.info("="*70)
    
    # if success_count == 0:
    #     logger.error("❌ TIDAK ADA FILE BERHASIL DIPROSES - PERIKSA LOG UNTUK DETAIL")
    #     #sys.exit(1)
    # elif success_count < len(clean_paths):
    #     logger.warning(f"⚠ {len(clean_paths) - success_count} file gagal diproses - lihat log untuk diagnosa")
    #     #sys.exit(0)
    # else:
    #     logger.info("✅ SEMUA FILE BERHASIL DIPROSES DENGAN PENANGANAN PARAMETER FLEKSIBEL")
    #     sys.exit(0)

2026-02-04 22:05:59,551 | INFO     | ======================================================================
2026-02-04 22:05:59,553 | INFO     | MEMULAI PROSES QC HARIAN - PENANGANAN PARAMETER TIDAK TERSEDIA
2026-02-04 22:05:59,554 | INFO     |   • Parameter dianggap 'tidak tersedia' jika SEMUA nilai = NaN
2026-02-04 22:05:59,555 | INFO     |   • QC hanya dijalankan untuk parameter dengan minimal 1 nilai valid
2026-02-04 22:05:59,555 | INFO     |   • Suhu: IQR x 3.0 | Curah Hujan: Hybrid (P99.9 + 600mm)
2026-02-04 22:05:59,556 | INFO     | Input directory : ../01.data/01.Obs_Homo
2026-02-04 22:05:59,557 | INFO     | QC output       : ../01.data/02.Obs_HybridIQR
2026-02-04 22:05:59,558 | INFO     | Report directory: ../01.data/03.Report
2026-02-04 22:05:59,558 | INFO     | Log file        : ../01.data/03.Report/qc_processing.log
2026-02-04 22:05:59,559 | INFO     | ======================================================================
2026-02-04 22:05:59,562 | INFO     | Ditemukan 110 f

SystemExit: 0

/home/api/anaconda3/envs/wrfpython/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
